In [ ]:
import kagglehub

path = kagglehub.dataset_download("adisongoh/it-service-ticket-classification-dataset")

print(path)

In [ ]:
import pandas as pd
import spacy
import re

df = pd.read_csv("all_tickets_processed_improved_v3.csv")

# if fail python -m spacy download en_core_web_sm
nlp = spacy.load("en_core_web_sm")

# define a cleaning function
def clean_text(text):
    # remove non letter
    text = re.sub(r"[^a-zA-Z\s]", "", text)
    text = text.lower()
    doc = nlp(text)
    # lemmatize and remove stopwords/punctuation/single characters
    tokens = [token.lemma_ for token in doc if token.is_alpha and not token.is_stop and len(token) > 1]
    return " ".join(tokens)


df["cleaned_text"] = df["Document"].apply(clean_text)
print(df[["Document", "cleaned_text", "Topic_group"]].head())


In [ ]:
texts = df["Document"].fillna("").tolist()

# batch process with nlp.pipe => faster than apply 4 times
cleaned_texts = []
for doc in nlp.pipe(texts, batch_size=50, n_process=4):
    # lowercase and remove non-letter characters
    tokens = [token.lemma_ for token in doc if token.is_alpha and not token.is_stop and len(token) > 1]
    cleaned_texts.append(" ".join(tokens))

# add back to dataframe
df["cleaned_text_bis"] = cleaned_texts

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer


vectorizer = TfidfVectorizer(max_features=5000)  # You can tweak max_features
X = vectorizer.fit_transform(df["cleaned_text"])

# labels
y = df["Topic_group"]

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# train model
clf = LogisticRegression(max_iter=1000)
clf.fit(X_train, y_train)

# evaluate
y_pred = clf.predict(X_test)
print(classification_report(y_test, y_pred))


In [ ]:
import joblib

# save classifier and vectorizer
joblib.dump(clf, "ticket_classifier_api/model/classifier.pkl")
joblib.dump(vectorizer, "ticket_classifier_api/model/vectorizer.pkl")
